# **Phase 3 - Generate Embeddings & Indexing**

In [1]:
%pip install -qU sentence-transformers faiss-cpu pandas pyarrow numpy tqdm joblib underthesea


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

import gc
import random
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from joblib import Parallel, delayed

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

import faiss
from sentence_transformers import SentenceTransformer
from underthesea import word_tokenize

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [3]:
DATA_DIR      = Path('workspace/data/cleaned')
PROCESSED_DIR = Path('workspace/data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

corpus      = pd.read_parquet(DATA_DIR / 'corpus.parquet')
val_split   = pd.read_parquet(DATA_DIR / 'val_split.parquet')
train_split = pd.read_parquet(DATA_DIR / 'train_split.parquet')

corpus_texts    = corpus['text'].tolist()
val_questions   = val_split['question'].tolist()
train_questions = train_split['question'].tolist()

## **1. Generate Embeddings & FAISS Index**

In [4]:
def get_embeddings(model, texts):
    with torch.no_grad(): 
        return model.encode(
            texts, 
            device               = 'cuda:0',
            batch_size           = 1024, 
            show_progress_bar    = True,
            normalize_embeddings = True,
            convert_to_numpy     = True
        )

def get_embeddings_and_faiss_index(model_path, prefix):
    model = SentenceTransformer(
        model_path, 
        model_kwargs={
            'torch_dtype'        : torch.bfloat16, 
            'attn_implementation': 'sdpa'
        }
    )
    model.max_seq_length = 512
    model.eval()

    corpus_embeds = get_embeddings(model, corpus_texts)
    val_embeds    = get_embeddings(model, val_questions)
    train_embeds  = get_embeddings(model, train_questions)

    np.save(PROCESSED_DIR / f"{prefix}_corpus_embeddings.npy", corpus_embeds)
    np.save(PROCESSED_DIR / f"{prefix}_val_embeddings.npy", val_embeds)
    np.save(PROCESSED_DIR / f"{prefix}_train_embeddings.npy", train_embeds)

    faiss_dim                       = corpus_embeds.shape[1]
    faiss_index                     = faiss.IndexHNSWFlat(faiss_dim, 32, faiss.METRIC_INNER_PRODUCT)
    faiss_index.hnsw.efConstruction = 200
    faiss_index.hnsw.efSearch       = 64
    faiss_index.add(corpus_embeds.astype(np.float32))

    faiss.write_index(faiss_index, str(PROCESSED_DIR / f"{prefix}_faiss_index.bin"))

    del model
    gc.collect()
    torch.cuda.empty_cache()

In [5]:
get_embeddings_and_faiss_index(
    model_path = 'AITeamVN/Vietnamese_Embedding', 
    prefix     = 'base'
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/708 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Batches:   0%|          | 0/231 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/95 [00:00<?, ?it/s]

In [6]:
get_embeddings_and_faiss_index(
    model_path = 'YuITC/vietnamese-embedding-vn-legal', 
    prefix     = 'ft'
)

modules.json:   0%|          | 0.00/429 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/282 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/241 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/90.0 [00:00<?, ?B/s]

Batches:   0%|          | 0/231 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/95 [00:00<?, ?it/s]

## **2. Generate BM25 Corpus**

In [7]:
def vn_tokenize(text): return word_tokenize(text, format='text').split()

corpus_tokens = Parallel(n_jobs=-1)(delayed(vn_tokenize)(text) for text in tqdm(corpus_texts, desc='Tokenizing corpus'))
val_tokens    = Parallel(n_jobs=-1)(delayed(vn_tokenize)(q)    for q    in tqdm(val_questions, desc='Tokenizing val queries'))
train_tokens  = Parallel(n_jobs=-1)(delayed(vn_tokenize)(q)    for q    in tqdm(train_questions, desc='Tokenizing train queries'))

Tokenizing corpus:   0%|          | 0/236199 [00:00<?, ?it/s]

Tokenizing val queries:   0%|          | 0/10717 [00:00<?, ?it/s]

Tokenizing train queries:   0%|          | 0/96453 [00:00<?, ?it/s]

In [8]:
pd.DataFrame({'tokens': corpus_tokens}).to_parquet(PROCESSED_DIR / 'bm25_corpus_tokens.parquet')
pd.DataFrame({'tokens': val_tokens}).to_parquet(PROCESSED_DIR / 'bm25_val_tokens.parquet')
pd.DataFrame({'tokens': train_tokens}).to_parquet(PROCESSED_DIR / 'bm25_train_tokens.parquet')